## Beispiel an MAGNIFICA HUMANITAS, OF HIS HOLINESS, POPE LEO XIV
Deutsch: https://www.vatican.va/content/leo-xiv/de/encyclicals/documents/20260515-magnifica-humanitas.html

Englisch: https://www.vatican.va/content/leo-xiv/en/encyclicals/documents/20260515-magnifica-humanitas.html

In [4]:
import requests
from bs4 import BeautifulSoup
import os
import re
from pprint import pprint
import json
import pandas as pd
import ast
import string

example_path = "example_data"
try:
    os.mkdir(example_path)
    print(f"Directory '{example_path}' created successfully.")
except FileExistsError:
    print(f"Directory '{example_path}' already exists.")

Directory 'example_data' already exists.


In [5]:
url = "https://www.vatican.va/content/leo-xiv/de/encyclicals/documents/20260515-magnifica-humanitas.html"

response = requests.get(url)
response.raise_for_status()

soup = BeautifulSoup(response.content, "html.parser")

# Remove script and style elements
for tag in soup(["script", "style"]):
    tag.decompose()

text = soup.get_text(separator="\n")

# Clean up blank lines
lines = [line.strip() for line in text.splitlines() if line]
start_index = lines.index("EINLEITUNG")
lines = lines[start_index:]
end_index = lines.index("Gegeben zu Rom, bei Sankt Peter, am 15. Mai des Jahres 2026, des zweiten meines Pontifikats.")+1
lines = [re.sub(r'\[\d+\]', '', l)  for l in lines[:end_index]]
lines = [x for x in lines if x.strip()]
text = " ".join(lines).split(".")
print(f"Extracted {len(text)} text segments from the document.")
pprint(text[:5])

output_path = f"{example_path}/magnifica_humanitas.json"
with open(rf"{output_path}", 'w') as f:
    json.dump(text, f)

print(f"Saved to {output_path} ({len(lines)} lines)")


Extracted 1542 text segments from the document.
['EINLEITUNG 1',
 ' Die von Gott geschaffene grossartige Menschheit steht heute vor einer '
 'entscheidenden Wahl: Entweder sie errichtet einen neuen Turm zu Babel oder '
 'sie erbaut die Stadt, in der Gott und die Menschheit gemeinsam wohnen',
 ' Jede Generation erbt die Aufgabe, die eigene Zeit zu gestalten, damit die '
 'Geschichte zu einem Ort reifen kann, an dem die Würde jedes Menschen '
 'gewahrt, Gerechtigkeit gefördert und Geschwisterlichkeit ermöglicht wird',
 ' Doch jeder Epoche droht die Gefahr, dass die Welt unmenschlich und '
 'ungerechter wird',
 ' Wo die Menschheit riskiert, ihr wahres Gesicht zu verlieren, da erheben wir '
 'Christen unseren Blick zu dem Gott, der Mensch geworden ist, in dem Wissen, '
 'dass sich »nur im Geheimnis des fleischgewordenen Wortes das Geheimnis des '
 'Menschen wahrhaft« aufklärt']
Saved to example_data/magnifica_humanitas.json (722 lines)


In [6]:
url_en = "https://www.vatican.va/content/leo-xiv/en/encyclicals/documents/20260515-magnifica-humanitas.html"

response = requests.get(url_en)
response.raise_for_status()

soup = BeautifulSoup(response.content, "html.parser")

for tag in soup(["script", "style"]):
    tag.decompose()
text_raw_en = soup.get_text(separator="\n")

lines_eng = [line.strip() for line in text_raw_en.splitlines() if line.strip()]
# "INTRODUCTION" appears twice (TOC + body); take the last occurrence
start_index = len(lines_eng) - 1 - lines_eng[::-1].index("INTRODUCTION")
lines_eng = lines_eng[start_index:]
end_index = next(i for i, l in enumerate(lines_eng) if l.startswith("Given in Rome, at Saint Peter")) + 1
lines_en = [re.sub(r'\[\d+\]', '', l) for l in lines_eng[:end_index]]
lines_en = [x for x in lines_en if x.strip()]
text_en = " ".join(lines_en).split(".")
print(f"Extracted {len(text_en)} text segments from the English document.")

output_path_en = f"{example_path}/magnifica_humanitas_en.json"
with open(rf"{output_path_en}", 'w') as f:
    json.dump(text_en, f)

print(f"Saved to {output_path_en} ({len(lines_en)} lines)")

Extracted 1583 text segments from the English document.
Saved to example_data/magnifica_humanitas_en.json (945 lines)


## Filtering 

In [7]:
# preprocessing auxiliary functions

stopwords = open("config_files/spacy_stopwords.txt").read().replace("{", '[').replace("}",']')
stopwords = ast.literal_eval(stopwords)

# this one is not needed if working on Sentence_spacy
def remove_punctuation(corpus):
    cleaned_corpus = corpus.translate(str.maketrans('', '', string.punctuation))
    return(cleaned_corpus)

# remove stop words from text
def remove_stop_words_from_text(cleaned_corpus):
    out = [word for word in cleaned_corpus if not word in stopwords]
    out_text = ' '.join(out)
    return out_text  # , out

# AI-terms
ki_terms = open(f"config_files/ki_terms.txt", "r").read()
ki_terms = [x for x in ki_terms.split("'") if x.strip()]
regex_ki = "".join(["\\b"+x.replace(" ","\\s")+"\\b|" for x in ki_terms])[:-1]
print(regex_ki)

ai_terms = open(f"config_files/ai_terms.txt", "r").read()
ai_terms = [x for x in ai_terms.split("'") if x.strip()]
regex_ai = "".join(["\\b"+x.replace(" ","\\s")+"\\b|" for x in ai_terms])[:-1]
print(regex_ai)

\bpalm\b|\blms\b|\bllama\b|\bTransformer\b|\bSprachmodelle\b|\bSprachmodell\b|\bgpt\b|\bplms\b|\bvortrainierte\sSprachmodelle\b|\bgpt-2\b|\bxlnet\b|\bgroße\sSprachmodelle\b|\bllms\b|\bgpt-3\b|\bStiftungsmodell\b|\bgpt-neo\b|\bgpt-j\b|\bchatgpt\b|\bgpt-4\b|\bKI\b|\bKünstliche\sIntelligenz\b|\bChatGPT\b|\bAI\b|\bCopilot\b|\bGemini\b|\bChatbot\b|\bLLM\b|\bChat-GPT\b|\bSprachmodell\b|\bKI-Modell\b|\bKI-System\b|\bKI-Tool\b|\bArtificial\sIntelligence\b|\bKI-Anwendung\b
\bpalm\b|\blms\b|\bllama\b|\bTransformern\b|\bLanguage\smodels\b|\bLanguage\smodel\b|\bgpt\b|\bplms\b|\bpre-trained\slanguage\smodels\b|\bgpt-2\b|\bxlnet\b|\blarge\slanguage\smodels\b|\bllms\b|\bgpt-3\b|\bFoundation\smodel\b|\bgpt-neo\b|\bgpt-j\b|\bchatgpt\b|\bgpt-4\b|\bAI\b|\bArtificial\sintelligence\b|\bChatGPT\b|\bAI\b|\bCopilot\b|\bGemini\b|\bChatbot\b|\bLLM\b|\bChat\sGPT\b|\bLanguage\smodel\b|\bAI\smodel\b|\bAI\ssystem\b|\bAI\stool\b|\bArtificial\sIntelligence\b|\bAI\sapplication\b


In [8]:
sentences_ger = [remove_punctuation(line.lower()) for line in text if re.search(regex_ki, line)]
sentences_en = [remove_punctuation(line.lower()) for line in text_en if re.search(regex_ai, line)]
print(f"Found {len(sentences_ger)} sentences in German and {len(sentences_en)} sentences in English containing KI terms.")

#save
with open(r'example_data/German_KI_sentences.json', 'w') as f:
    json.dump(sentences_ger, f)
with open(r"example_data/English_KI_sentences.json", "w") as f:
        json.dump(sentences_en, f)

Found 66 sentences in German and 62 sentences in English containing KI terms.


## AnthroScore 

In [9]:
# generate input string for AnthroScore for both languages


# read results

# plot results

## RegEx and SemEx

In [10]:
# import methods from Anthrop_search.ipynb

# run on sentences from german encyclical


In [11]:
# plot results as in Anthrop_search.ipynb (import mehtods to plot)

## Run trhough models MetaphorBERT and SentenceMetaphorBERT

In [12]:
# import models and methods to run on sentences from german encyclical

In [13]:
# try to run BasicBERT on englisch encyclical (AI and non-AI seperated)